# Week 21: High-Performance ML Inference, Compilers & Serving
## Notebook 05: OpenAI Triton GPU Programming Fundamentals from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Historically, writing custom high-performance GPU kernels required raw **CUDA C/C++**:
- Managing individual GPU hardware threads (32 threads per warp, thread blocks, shared memory synchronization, memory coalescing).
- Writing CUDA is notoriously difficult, error-prone, and takes weeks of specialized engineering.

In 2021, Philippe Tillet (OpenAI) introduced **Triton**:
- A Python-based programming language and compiler.
- **Block-Level Programming Model:** Programmers operate on **Blocks of Tensors** (e.g. 128 elements), while the Triton compiler automatically handles warp synchronization, shared memory caching, and memory coalescing!
- Powers modern state-of-the-art implementations like **FlashAttention-2** and **Unsloth**!

---

### 2. Mathematical Foundations: The GPU Memory Hierarchy
1. **Global Memory (HBM/VRAM):** Large (80GB), but slow (~2 TB/s bandwidth).
2. **On-Chip Shared Memory / Cache (SRAM):** Tiny (~256 KB per Streaming Multiprocessor), but ultra-fast (~19 TB/s bandwidth).
3. **Registers:** Direct ALU storage (sub-nanosecond).

Triton's primary design pattern is **Tiling**:
Load blocks of $B$ elements from slow HBM into fast SRAM, perform fused computation, and write back to HBM.

In [ ]:
import numpy as np

# Step 1: Simulate the Triton Block-Level Programming Model in Python
# Vector Addition: Z = X + Y
N = 32
BLOCK_SIZE = 8  # Each GPU program instance processes a block of 8 elements

X = np.arange(N, dtype=float)
Y = np.ones(N, dtype=float) * 10.0
Z = np.zeros(N, dtype=float)

def triton_vector_add_kernel_simulation(x_ptr, y_ptr, z_ptr, n_elements, BLOCK_SIZE):
    """
    Simulates Triton grid execution where multiple Program IDs (PIDs)
    run concurrently on different Streaming Multiprocessors.
    """
    num_programs = (n_elements + BLOCK_SIZE - 1) // BLOCK_SIZE
    
    # In Triton: Each block is assigned a program_id = pid
    for pid in range(num_programs):
        # 1. Compute block offsets: offsets = pid * BLOCK_SIZE + arange(BLOCK_SIZE)
        offsets = pid * BLOCK_SIZE + np.arange(BLOCK_SIZE)
        
        # 2. Create boundary mask to prevent out-of-bounds memory access
        mask = offsets < n_elements
        
        # 3. Block-level memory load from global VRAM
        x_block = np.where(mask, x_ptr[offsets], 0.0)
        y_block = np.where(mask, y_ptr[offsets], 0.0)
        
        # 4. Compute in on-chip SRAM/registers
        z_block = x_block + y_block
        
        # 5. Block-level memory store back to global VRAM
        for idx, offset in enumerate(offsets):
            if mask[idx]:
                z_ptr[offset] = z_block[idx]

triton_vector_add_kernel_simulation(X, Y, Z, N, BLOCK_SIZE)

print("--- Simulated Triton Block Execution ---")
print("Input X:\n", X[:16])
print("Input Y:\n", Y[:16])
print("Output Z (X + Y):\n", Z[:16])
print("Verification Exact Match:", np.allclose(Z, X + Y))

### 3. Step-by-Step Code Walkthrough
1. **Grid Partitioning:** $N=32$ elements divided by `BLOCK_SIZE=8` yields 4 parallel program instances.
2. **Offset Arithmetic:** Each block instance computes its slice indices: `pid * BLOCK_SIZE + arange(BLOCK_SIZE)`.
3. **Masking:** Protects against reading memory past the end of the array.

---

### 4. Industry Applications & Trade-offs
- **FlashAttention-2 & Liger-Kernel:** Written entirely in Triton, matching or beating hand-crafted CUDA C++ kernels with 5x fewer lines of code.
- **Portability:** Compiles to NVIDIA PTX as well as AMD ROCm GPU assembly.

---

### 5. Top Interview Questions & Answers
**Q1: How does Triton differ from CUDA C++ in its programming abstraction?**
> **Answer:** In CUDA, the programmer must explicitly manage individual threads, warp shuffles, shared memory banks, and barrier synchronizations (`__syncthreads()`). In Triton, the programmer writes code at the level of tensor blocks (e.g. 64x64 matrices), and the compiler automatically optimizes thread scheduling and register allocation.

**Q2: What is memory coalescing on modern GPUs?**
> **Answer:** When consecutive threads within a 32-thread warp access consecutive 32-bit memory addresses in physical VRAM, the GPU hardware consolidates those 32 requests into a single 128-byte cache line transaction. Non-contiguous access causes 32 separate transactions, slashing memory throughput by 90%.